# Retrieval poisoning

Here a **retriever** chooses what the model sees, as in a real RAG system.

- **Collection:** all paragraphs of the run's questions (each once). For every poisoned question, the original answer paragraph is
  replaced by its 4 versions at level m (m false, 4 − m true).
- **Retrievers:** BM25 (sparse, word overlap) and dense (cosine similarity of `nomic-embed-text` embeddings).
- **Top k = 5**, the same number of paragraphs as in the controlled experiment; given to the model in rank order.

Two questions: does the poison **get in** (how many false versions are retrieved), and does it **win** (what the model answers)?

> **Note (17 Nov 2026).** The tables in this notebook use the scoring stored at run time (strict `contains_answer`) and, where known / unknown appears, the closed-book labels of notebook 01. The numbers reported in the paper come from `06_analysis.ipynb`, which re-scores the same saved answers with the lenient rule and uses the notebook 00 labels. The conclusions are the same; single numbers differ slightly.

In [ ]:
import json
from pathlib import Path

import pandas as pd

from ragtrust.runs import load_selection
from ragtrust.analysis import keep_correct_at_m0, share_by_m
from ragtrust.data import load_hotpotqa, pick_fewshot_examples
from ragtrust.experiment import cached_run, run_retrieval
from ragtrust.llm import OllamaLLM
from ragtrust.poison import load_poison_set, select_items
from ragtrust.retrieval import OllamaEmbedder, build_corpus

MODELS = ["llama3.1:8b", "qwen2.5:7b"]
RETRIEVERS = ["bm25", "dense"]
K = 5
TAG = "final"   # part of every result file name
SEED = 0
RESULTS = Path("../results")
POISON_PATH = Path(f"../data/poison/gemma4-12b_{TAG}.jsonl")
EXCLUDED_PATH = Path("../data/poison/excluded.json")

examples = load_selection()
excluded = json.load(open(EXCLUDED_PATH)) if EXCLUDED_PATH.exists() else {}
items = select_items(load_poison_set(POISON_PATH), examples, excluded)
embedder = OllamaEmbedder("nomic-embed-text", cache_path=Path("../data/cache/nomic-embed-text.pkl"))
print(f"Poisoned questions: {len(items)} | collection size at m = 0: {len(build_corpus(examples, items, 0))}")

Poisoned questions: 751 | collection size at m = 0: 10130


## 1. Retrieval only: does the poison get in?

No generation, so this takes seconds (the first dense run embeds the collection once, then the vectors are cached).

- `n_false`: false versions in the top 5 (0..m)
- `n_true`: true versions in the top 5 (0..4 − m)
- `bridge_retrieved`: the other gold paragraph is in the top 5 (needed for the multi-hop question)

In [ ]:
retrieval = pd.concat([run_retrieval(None, examples, items, r, embedder, k=K) for r in RETRIEVERS])
retrieval.groupby(["retriever", "m"])[["n_false", "n_true", "bridge_retrieved"]].mean().round(2)

## 2. Full run: every model × retriever × m

In [ ]:
fewshot = pick_fewshot_examples(seed=SEED)
frames = []
for model in MODELS:
    llm = OllamaLLM(model=model, seed=SEED)
    for retriever in RETRIEVERS:
        path = RESULTS / f"retrieval_{retriever}_{model.replace(':', '-')}_{TAG}.csv"
        run = lambda: run_retrieval(llm, examples, items, retriever, embedder, k=K, fewshot=fewshot)
        frames.append(cached_run(path, llm, run).assign(model=model))
df = pd.concat(frames)

## 3. Results

Per model, the questions answered correctly at m = 0 with **both** retrievers.

In [4]:
flippable = keep_correct_at_m0(df)
print(flippable.drop_duplicates(["model", "id"])["model"].value_counts(), "\n")
share_by_m(flippable, ["model", "retriever"])

model
llama3.1:8b    425
qwen2.5:7b     399
Name: count, dtype: int64 



m                        0      1      2      3      4
model       retriever                                 
llama3.1:8b bm25       0.0  0.042  0.113  0.282  0.471
            dense      0.0  0.059  0.165  0.355  0.635
qwen2.5:7b  bm25       0.0  0.038  0.178  0.396  0.571
            dense      0.0  0.073  0.216  0.461  0.694

**The effective dose.** The retriever decides how many false versions the model actually sees.
Grouping by `n_false` (false versions retrieved) instead of m shows how much each retrieved false version weighs.

In [5]:
flippable.groupby(["model", "retriever", "n_false"])["category"].agg(
    poisoned=lambda c: (c == "poisoned").mean(), n="size").round(3)

poisoned     n
model       retriever n_false                
llama3.1:8b bm25      0           0.002  1101
                      1           0.192   411
                      2           0.298   275
                      3           0.584   209
                      4           0.783   129
            dense     0           0.004   705
                      1           0.112   455
                      2           0.272   393
                      3           0.506   338
                      4           0.786   234
qwen2.5:7b  bm25      0           0.004   976
                      1           0.204   407
                      2           0.449   276
                      3           0.715   207
                      4           0.876   129
            dense     0           0.008   656
                      1           0.142   431
                      2           0.351   368
                      3           0.611   319
                      4           0.842   221